In [2]:
import sys
from pathlib import Path

# Adjust path to find the 'src' directory
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Lab 13: Apple Brand Monitoring Dash Dashboard

This notebook documents the Dash dashboard implementation for the Apple social media brand monitoring pipeline. The dashboard uses the cleaned pipeline dataset, reads from MongoDB when available, falls back to the cleaned CSV when MongoDB is unavailable, and can be run locally or with Docker Compose.


## Dashboard Structure

- `app.py` creates the Dash app, applies Bootstrap, loads the layout, registers callbacks, and exposes `server` for Gunicorn.
- `src/dashboard/layout.py` defines the dashboard page with KPI cards, filters, charts, a recent mentions table, and a live ticker.
- `src/dashboard/callbacks.py` contains the interactive callbacks for KPI cards, four charts, the recent mentions table, and the live ticker.
- `src/dashboard/data_access.py` contains MongoDB access, CSV fallback logic, and filtering helpers.
- `scripts/seed_mongo.py` loads the cleaned Apple mention dataset into MongoDB.


In [3]:
from src.dashboard.data_access import load_mentions, get_source_options, get_document_type_options, get_year_bounds

df = load_mentions()
print(df.shape)
print(get_source_options(df)[:5])
print(get_document_type_options(df))
print(get_year_bounds(df))

(101, 22)
[{'label': 'All sources', 'value': 'All'}, {'label': 'apple_ratings_large.csv', 'value': 'apple_ratings_large.csv'}, {'label': 'newsapi_Apple_page_1.json', 'value': 'newsapi_Apple_page_1.json'}, {'label': 'newsapi_Apple_page_2.json', 'value': 'newsapi_Apple_page_2.json'}, {'label': 'newsapi_Apple_page_3.json', 'value': 'newsapi_Apple_page_3.json'}]
[{'label': 'All document types', 'value': 'All'}, {'label': 'csv', 'value': 'csv'}, {'label': 'json', 'value': 'json'}, {'label': 'xml', 'value': 'xml'}]
(2026, 2026)


## Callback Behavior

The dashboard includes shared filters for source, document type, year range, and text search. The same filters update the source volume bar chart, rating distribution chart, rating-over-time scatter chart, yearly trend chart, KPI cards, and recent mentions table. The live ticker uses `dcc.Interval` and refreshes automatically every three seconds.


## Local Run

Install the dashboard packages and start the app:

```bash
pip install dash dash-bootstrap-components pymongo gunicorn websockets
python app.py
```

Open `http://127.0.0.1:8050` and verify that the Apple dashboard loads with filters, KPI cards, charts, a recent mentions table, and the live ticker.


## Deployment Process

### Prerequisites

- Docker Desktop must be installed and running.
- Ports `8050` and `27017` should be available on the local machine.
- The cleaned dataset should exist at `data/processed/cleaned/cleaned_data.csv`.

### Start the stack and seed MongoDB

From the project root, start the Dash app and MongoDB:

```bash
docker compose up --build
```

In a second terminal, seed MongoDB with the cleaned Apple brand monitoring dataset:

```bash
python scripts/seed_mongo.py
```

If running the seed script from the host while MongoDB is exposed by Docker Compose, the default `mongodb://localhost:27017` connection string works. Inside Docker Compose, the web service uses `mongodb://db:27017`.

### Verify the app

Open `http://localhost:8050`. The dashboard should show the Apple Brand Intelligence Dashboard with KPI cards, source and document type filters, a year slider, search input, four charts, a recent mentions table, and a live ticker that updates every few seconds. Changing any filter should update all filtered charts.

### Stop the stack

Stop containers cleanly with:

```bash
docker compose down
```

To also remove the MongoDB volume and clear stored records, run:

```bash
docker compose down -v
```

### Troubleshooting

If port `8050` is already in use, either stop the other process or change the Compose port mapping from `8050:8050` to another host port such as `8051:8050`, then open `http://localhost:8051`. If MongoDB is not seeded yet, the dashboard still loads using the cleaned CSV fallback.
